## Quality of Care Report

This report displays a compact year-level summary of quality-of-care indicators and points to generated map outputs.

In [ ]:
ROOT_PATH <- "~/workspace"
CONFIG_PATH <- file.path(ROOT_PATH, "configuration")
CODE_PATH <- file.path(ROOT_PATH, "code")
DATA_PATH <- file.path(ROOT_PATH, "data", "dhis2", "quality_of_care")
FIGURES_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "reporting", "outputs", "figures")

source(file.path(CODE_PATH, "snt_utils.r"))
install_and_load(c("jsonlite", "data.table", "arrow", "dplyr", "knitr", "glue", "reticulate", "writexl", "ggplot2", "scales", "gridExtra", "sf"))

# Create output directories
REPORT_OUTPUTS_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "reporting", "outputs")
dir.create(REPORT_OUTPUTS_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

Sys.setenv(RETICULATE_PYTHON = "/opt/conda/bin/python")
openhexa <- reticulate::import("openhexa.sdk")

config_json <- jsonlite::fromJSON(file.path(CONFIG_PATH, "SNT_config.json"))
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

In [ ]:
# Use district-year output file (latest action)
files <- list.files(DATA_PATH, pattern = paste0("^", COUNTRY_CODE, "_quality_of_care_district_year_(imputed|removed)\\.parquet$"), full.names = TRUE)
if (length(files) == 0) {
  stop(glue::glue("No quality_of_care parquet found in {DATA_PATH}"))
}

latest_file <- files[which.max(file.info(files)$mtime)]
qoc <- as.data.table(arrow::read_parquet(latest_file))

# Build summary table with only available columns
# Start with unique YEAR values
summary_tbl <- unique(qoc[, .(YEAR)])

# Add rate indicators (mean) - merge one by one
if ("testing_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(testing_rate = mean(testing_rate, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}
if ("treatment_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(treatment_rate = mean(treatment_rate, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}
if ("case_fatality_rate" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(case_fatality_rate = mean(case_fatality_rate, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}
if ("prop_adm_malaria" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(prop_adm_malaria = mean(prop_adm_malaria, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}
if ("prop_malaria_deaths" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(prop_malaria_deaths = mean(prop_malaria_deaths, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}

# Add absolute indicators (sum)
if ("non_malaria_all_cause_outpatients" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(non_malaria_all_cause_outpatients = sum(non_malaria_all_cause_outpatients, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}
if ("presumed_cases" %in% names(qoc)) {
  summary_tbl <- merge(summary_tbl, 
                       qoc[, .(presumed_cases = sum(presumed_cases, na.rm = TRUE)), by = .(YEAR)], 
                       by = "YEAR", all.x = TRUE)
}

summary_tbl <- summary_tbl[order(YEAR)]

# Explicitly list missing indicators so report is self-explanatory
expected_indicators <- c(
  "testing_rate",
  "treatment_rate",
  "case_fatality_rate",
  "prop_adm_malaria",
  "prop_malaria_deaths",
  "non_malaria_all_cause_outpatients",
  "presumed_cases"
)
missing_indicators <- setdiff(expected_indicators, names(qoc))
if (length(missing_indicators) > 0) {
  log_msg(glue::glue("[WARNING] Missing indicators in input file: {paste(missing_indicators, collapse=', ')}"), level = "warning")
  cat(glue::glue("\nMissing indicators in this run: {paste(missing_indicators, collapse=', ')}\n"))
  cat("Reason: required source columns are absent in the selected outliers file.\n")
}

# Save summary data (parquet, csv, xlsx) - following other pipelines pattern
summary_parquet <- file.path(REPORT_OUTPUTS_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_summary.parquet"))
summary_csv <- file.path(REPORT_OUTPUTS_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_summary.csv"))
summary_xlsx <- file.path(REPORT_OUTPUTS_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_summary.xlsx"))

# Save as parquet (primary format, like other pipelines)
arrow::write_parquet(summary_tbl, summary_parquet)

# Save as csv and xlsx for compatibility
data.table::fwrite(summary_tbl, summary_csv)
writexl::write_xlsx(list(summary = as.data.frame(summary_tbl)), summary_xlsx)

log_msg(glue::glue("Summary data saved to: {summary_parquet}, {summary_csv}, {summary_xlsx}"))

knitr::kable(summary_tbl, caption = "Quality of Care - Year-level summary")

cat(glue::glue("\nLoaded file: {latest_file}\n"))
cat(glue::glue("Map outputs folder: {FIGURES_PATH}\n"))
cat(glue::glue("Summary data saved to: {summary_parquet}, {summary_csv}, {summary_xlsx}\n"))

## Graphs by Year

In [ ]:
# Create bar charts by year (same as original notebook - 4x2 grid layout)
# Prepare data - convert rates to percentages
plot_data <- copy(summary_tbl)

# Create the same 4x2 subplot layout as original notebook
if (nrow(plot_data) > 0) {
  # Create a list to store individual plots (in order: 4x2 grid)
  plots_list <- list()
  
  # Row 0, Col 0: Testing rate
  if ("testing_rate" %in% names(plot_data)) {
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = testing_rate * 100)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = paste0(round(testing_rate * 100, 1), "%")), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Testing rate (TEST / SUSP)", x = "Année", y = "%") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.1)))
    plots_list[["testing_rate"]] <- p
  }
  
  # Row 0, Col 1: Treatment rate
  if ("treatment_rate" %in% names(plot_data)) {
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = treatment_rate * 100)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = paste0(round(treatment_rate * 100, 1), "%")), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Treatment rate (MALTREAT / CONF)", x = "Année", y = "%") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.1)))
    plots_list[["treatment_rate"]] <- p
  }
  
  # Row 1, Col 0: Case fatality rate
  if ("case_fatality_rate" %in% names(plot_data)) {
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = case_fatality_rate * 100)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = paste0(round(case_fatality_rate * 100, 1), "%")), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Case fatality rate (MALDTH / MALADM)", x = "Année", y = "%") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.1)))
    plots_list[["case_fatality_rate"]] <- p
  }
  
  # Row 1, Col 1: Proportion admissions malaria
  if ("prop_adm_malaria" %in% names(plot_data)) {
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = prop_adm_malaria * 100)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = paste0(round(prop_adm_malaria * 100, 1), "%")), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Prop. admissions paludisme (MALADM / ALLADM)", x = "Année", y = "%") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.1)))
    plots_list[["prop_adm_malaria"]] <- p
  }
  
  # Row 2, Col 0: Proportion deaths malaria
  if ("prop_malaria_deaths" %in% names(plot_data)) {
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = prop_malaria_deaths * 100)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = paste0(round(prop_malaria_deaths * 100, 1), "%")), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Prop. décès paludisme (MALDTH / ALLDTH)", x = "Année", y = "%") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(expand = expansion(mult = c(0, 0.1)))
    plots_list[["prop_malaria_deaths"]] <- p
  }
  
  # Row 2, Col 1: Presumed cases (absolute)
  if ("presumed_cases" %in% names(plot_data)) {
    format_label <- function(v) {
      ifelse(is.na(v) | v == 0, "0",
        ifelse(v >= 1e6, paste0(round(v/1e6, 2), "M"),
          format(round(v), big.mark = " ", scientific = FALSE)
        )
      )
    }
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = presumed_cases)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = format_label(presumed_cases)), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Cas présumés (PRES)", x = "Année", y = "Nombre") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(labels = scales::comma, expand = expansion(mult = c(0, 0.1)))
    plots_list[["presumed_cases"]] <- p
  }
  
  # Row 3, Col 0: Non-malaria all-cause outpatients (absolute)
  if ("non_malaria_all_cause_outpatients" %in% names(plot_data)) {
    format_label <- function(v) {
      ifelse(is.na(v) | v == 0, "0",
        ifelse(v >= 1e6, paste0(round(v/1e6, 2), "M"),
          format(round(v), big.mark = " ", scientific = FALSE)
        )
      )
    }
    p <- ggplot(plot_data, aes(x = factor(YEAR), y = non_malaria_all_cause_outpatients)) +
      geom_bar(stat = "identity", fill = "#2563eb", color = "#1e40af", width = 0.7) +
      geom_text(aes(label = format_label(non_malaria_all_cause_outpatients)), 
                vjust = -0.5, size = 2.5) +
      labs(title = "Consultations externes non-paludisme (ALLOUT)", x = "Année", y = "Nombre") +
      theme_minimal() +
      theme(
        plot.title = element_text(face = "bold", size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1, size = 9),
        panel.grid.major.y = element_line(linetype = "dashed", color = scales::alpha("grey", 0.7)),
        plot.background = element_rect(fill = "#fafafa", color = NA),
        panel.background = element_rect(fill = "#fafafa", color = NA),
        plot.margin = margin(5, 5, 5, 5)
      ) +
      scale_y_continuous(labels = scales::comma, expand = expansion(mult = c(0, 0.1)))
    plots_list[["non_malaria_all_cause_outpatients"]] <- p
  }
  
  # Create and display combined plot (dynamic grid for readability)
  if (length(plots_list) > 0) {
    # Order plots as in original
    plot_order <- c("testing_rate", "treatment_rate", "case_fatality_rate", "prop_adm_malaria", 
                    "prop_malaria_deaths", "presumed_cases", "non_malaria_all_cause_outpatients")
    available_plots <- plots_list[intersect(plot_order, names(plots_list))]

    if (length(available_plots) > 0) {
      n_plots <- length(available_plots)
      ncol_layout <- 2
      nrow_layout <- ceiling(n_plots / ncol_layout)

      # Bigger display in report so labels are readable
      options(repr.plot.width = 14, repr.plot.height = max(7, 4.8 * nrow_layout))

      combined_plot <- do.call(grid.arrange, c(available_plots, ncol = ncol_layout, nrow = nrow_layout))
      print(combined_plot)

      # Save at larger size for presentation readability
      combined_file <- file.path(FIGURES_PATH, glue::glue("{COUNTRY_CODE}_quality_of_care_by_year.png"))
      ggsave(
        combined_file,
        plot = combined_plot,
        width = 18,
        height = max(8, 5.2 * nrow_layout),
        dpi = 300,
        bg = "white",
        units = "in"
      )
      log_msg(glue::glue("Combined bar charts saved: {combined_file}"))
    }
  }
}

## Maps by District and Year

Maps are generated directly from the quality-of-care data and district shapes.

In [ ]:
# Load shapes geojson from dataset (like seasonality pipeline)
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

shapes <- tryCatch({
  get_latest_dataset_file_in_memory(DHIS2_FORMATTED_DATASET, paste0(COUNTRY_CODE, "_shapes.geojson"))
}, error = function(e) {
  msg <- paste0("Error while loading DHIS2 Shapes data for: ", COUNTRY_CODE, ". ", conditionMessage(e))
  log_msg(msg, level = "error")
  stop(msg)
})

# Ensure ADM2_ID is character in both datasets
shapes$ADM2_ID <- as.character(shapes$ADM2_ID)
qoc$ADM2_ID <- as.character(qoc$ADM2_ID)

# Merge shapes with quality-of-care data
qoc_sf <- shapes %>%
  dplyr::left_join(qoc, by = "ADM2_ID")

# Helper to build readable interval labels for legends
format_interval_labels <- function(breaks_vec) {
  labels <- c()
  for (i in seq_len(length(breaks_vec) - 1)) {
    a <- breaks_vec[i]
    b <- breaks_vec[i + 1]
    labels <- c(labels, paste0(scales::comma(round(a)), " - ", scales::comma(round(b))))
  }
  labels
}

# Function to plot yearly maps (similar to code notebook but inline in report)
plot_yearly_map_report <- function(sf_data, value_col, title_prefix, is_rate = TRUE) {
  if (!(value_col %in% names(sf_data))) {
    log_msg(glue::glue("[WARNING] Column '{value_col}' not found. Skipping map generation."), level = "warning")
    return(invisible(NULL))
  }
  
  years <- sort(unique(sf_data$YEAR[!is.na(sf_data$YEAR)]))
  if (length(years) == 0) {
    log_msg(glue::glue("[WARNING] No valid years for '{value_col}'. Skipping map."), level = "warning")
    return(invisible(NULL))
  }
  
  # Create plots for each year
  plot_list <- list()
  base_shapes <- sf_data %>% dplyr::select(ADM2_ID, geometry) %>% dplyr::distinct()

  for (yr in years) {
    # Keep all districts on map, then join year values
    year_vals <- sf_data[sf_data$YEAR == yr, c("ADM2_ID", value_col), drop = FALSE]
    year_vals <- sf::st_drop_geometry(year_vals)
    year_vals <- year_vals[!duplicated(year_vals$ADM2_ID), , drop = FALSE]
    sf_y <- dplyr::left_join(base_shapes, year_vals, by = "ADM2_ID")

    vals <- sf_y[[value_col]]
    finite_vals <- vals[is.finite(vals) & !is.na(vals)]

    if (length(finite_vals) == 0) {
      next
    }

    # Create categories
    if (is_rate) {
      cat_vals <- cut(
        vals,
        breaks = c(-Inf, 0, 0.2, 0.4, 0.6, 0.8, 1.0, Inf),
        labels = c("<0", "0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1.0", ">1.0"),
        include.lowest = TRUE
      )
      fill_palette <- "YlOrRd"
    } else {
      # Use readable fixed-count classes for absolute values
      n_classes <- 5
      br <- unique(as.numeric(quantile(finite_vals, probs = seq(0, 1, length.out = n_classes + 1), na.rm = TRUE)))
      br <- sort(br)
      if (length(br) < 2) {
        br <- c(min(finite_vals, na.rm = TRUE), max(finite_vals, na.rm = TRUE) + 1)
      }
      if (length(unique(br)) < 2) {
        cat_vals <- as.factor(rep("single value", nrow(sf_y)))
      } else {
        labels_abs <- format_interval_labels(br)
        cat_vals <- cut(vals, breaks = br, include.lowest = TRUE, labels = labels_abs)
      }
      fill_palette <- "Blues"
    }

    sf_y$cat <- as.factor(cat_vals)

    p <- ggplot(sf_y) +
      geom_sf(aes(fill = cat), color = "grey60", size = 0.12) +
      scale_fill_brewer(palette = fill_palette, na.value = "#f3f4f6", drop = FALSE) +
      theme_void() +
      labs(
        title = paste0(title_prefix, " - ", yr),
        fill = ifelse(is_rate, "Rate class", "Value class")
      ) +
      guides(fill = guide_legend(nrow = 2, byrow = TRUE)) +
      theme(
        legend.position = "bottom",
        legend.text = element_text(size = 9),
        legend.title = element_text(size = 10, face = "bold"),
        plot.title = element_text(face = "bold", size = 13)
      )

    plot_list[[as.character(yr)]] <- p
  }
  
  # Display all plots
  if (length(plot_list) > 0) {
    options(repr.plot.width = 10, repr.plot.height = 8)
    for (yr_name in names(plot_list)) {
      print(plot_list[[yr_name]])
    }
  }
}

# Generate maps for each available indicator
cat("### Testing Rate\n")
if ("testing_rate" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "testing_rate", "Testing rate (TEST / SUSP)", TRUE)
}

cat("\n### Treatment Rate\n")
if ("treatment_rate" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "treatment_rate", "Treatment rate (MALTREAT / CONF)", TRUE)
}

cat("\n### Case Fatality Rate\n")
if ("case_fatality_rate" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "case_fatality_rate", "In-hospital case fatality rate (MALDTH / MALADM)", TRUE)
}

cat("\n### Proportion Admissions Malaria\n")
if ("prop_adm_malaria" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "prop_adm_malaria", "Proportion admitted for malaria (MALADM / ALLADM)", TRUE)
}

cat("\n### Proportion Malaria Deaths\n")
if ("prop_malaria_deaths" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "prop_malaria_deaths", "Proportion of malaria deaths (MALDTH / ALLDTH)", TRUE)
}

cat("\n### Non-malaria All-cause Outpatients\n")
if ("non_malaria_all_cause_outpatients" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "non_malaria_all_cause_outpatients", "Non-malaria all-cause outpatients (ALLOUT)", FALSE)
}

cat("\n### Presumed Cases\n")
if ("presumed_cases" %in% names(qoc_sf)) {
  plot_yearly_map_report(qoc_sf, "presumed_cases", "Presumed cases (PRES)", FALSE)
}